# TamilTech-QA — End-to-End Pipeline on Kaggle T4 (v2 — All Fixes Baked In)

This v2 includes ALL the patches we discovered through trial-and-error in the previous session:
- ✅ Deep recursive repo path discovery (handles nested Kaggle paths)
- ✅ OpenAI gpt-4o-mini with JSON mode (no parse failures)
- ✅ Fast exact dedup (skip slow MinHash)
- ✅ Modern trl API (SFTConfig + processing_class)
- ✅ bf16 training (no fp16 grad scaler conflicts)
- ✅ Reduced batch size (fits T4 memory)
- ✅ Data backup step (prevents loss on session timeout)
- ✅ Smaller epochs (fits 12-hour session limit)

## Before running

1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. **Settings → Persistence → Files and variables**
4. **Add-ons → Secrets**: add `OPENAI_API_KEY`, `YOUTUBE_API_KEY`, `HF_TOKEN`
5. **+ Add Input → Datasets**: attach your `tamiltech-qa-repo` dataset

## Phase map

| Cell | Phase | ETA |
|---|---|---|
| 1-6 | Setup + imports + patches | ~5 min |
| 7-10 | Data collection + preprocess | ~50 min |
| 11 | 💾 SAVE DATA BACKUP | ~2 min |
| 12-14 | Training | ~5-6 hrs |
| 15-16 | Evaluation + push | ~30 min |

**Total: ~7 hours. SAVE VERSION every 30 min during training.**

## 1 — Setup repo + working dir

In [ ]:
import os, sys, shutil
from pathlib import Path

def find_repo_root(start='/kaggle/input', max_depth=6):
    """Recursively find folder containing src/ and config/."""
    for root, dirs, files in os.walk(Path(start)):
        if 'src' in dirs and 'config' in dirs:
            return Path(root)
    return None

REPO_INPUT = find_repo_root()
if REPO_INPUT is None:
    raise FileNotFoundError('No src/+config/ found under /kaggle/input/. Attach tamiltech-qa-repo dataset.')
print('Found repo at:', REPO_INPUT)

WORK = Path('/kaggle/working/tamiltech-qa')
if not WORK.exists():
    shutil.copytree(str(REPO_INPUT), str(WORK))
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print('Working dir:', os.getcwd())

## 2 — Install dependencies

In [ ]:
!pip install -q -U peft>=0.13.0 trl>=0.12.0 bitsandbytes>=0.44.0 \
    datasets>=3.1.0 accelerate>=1.1.0 transformers>=4.46.0 \
    google-api-python-client openai langdetect rouge_score bert_score datasketch \
    loguru python-dotenv huggingface_hub

In [ ]:
# NLTK
import nltk
for p in ('punkt', 'punkt_tab'):
    try: nltk.download(p, quiet=True)
    except: pass

# Load secrets from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ('OPENAI_API_KEY', 'YOUTUBE_API_KEY', 'HF_TOKEN', 'WANDB_API_KEY'):
    try:
        os.environ[k] = s.get_secret(k)
        print(f'{k}: set ({len(os.environ[k])} chars)')
    except:
        print(f'{k}: NOT set')
if 'WANDB_API_KEY' not in os.environ:
    os.environ['WANDB_DISABLED'] = 'true'
    os.environ['WANDB_MODE'] = 'disabled'

In [ ]:
# Sanity check
from src.utils import load_config, project_root
from src.utils.logger import setup_logging
from src.utils.seed import seed_everything
setup_logging(level='INFO')
seed_everything(42)
cfg = load_config('config/data_config.yaml')
print('OpenAI model:', cfg['synthetic']['model'])
print('Topics:', cfg['synthetic']['topics'])
print('Project root:', project_root())

## 3 — Apply ALL patches (configs + synthetic JSON + fast dedup)

In [ ]:
import yaml, json
from googleapiclient.discovery import build

# Resolve The AI Dude
yt = build('youtube', 'v3', developerKey=os.environ['YOUTUBE_API_KEY'], cache_discovery=False)
ai_dude_id = None
try:
    r = yt.channels().list(part='id', forHandle='@TheAIDude-Tamil').execute()
    if r.get('items'):
        ai_dude_id = r['items'][0]['id']
        print(f'Resolved AI Dude: {ai_dude_id}')
except Exception as e:
    print(f'AI Dude resolution failed (continuing): {e}')

# Data config: 12 channels + loose filters
dc = yaml.safe_load(open('config/data_config.yaml'))
dc['youtube']['target_channels'] = [
    'UCIFQgj1Rhx-FFgyo0zzPSfw',
    'UCKOob5-7sMljgW3f4pO_Dyg',
    'UCEKv3WR3HUVuIHV2eXFyGYg',
    'UC20sXo8ReewkzNKBFgzVCPA',
    'UCvyZS6W6zMJCZBVzF-Ei6sw',
    'UCmCAY_mStg1POKUWgMg-aGQ',
    'UCY8kgTLO7GitoKuxz4Cw3uQ',
    'UCBF5i6PogoMwnoAP0LFiCmQ',
    'UC9xghV-TcBwGvK-aEMhpt5w',
    'UCApUMSkgDT8ayJZU8jBweYw',
    'UCvhU9qF1xtUsFXdKrcJxbFA',
]
if ai_dude_id:
    dc['youtube']['target_channels'].append(ai_dude_id)
dc['youtube']['max_videos_per_channel'] = 80
dc['youtube']['max_comments_per_video'] = 200
dc['synthetic']['model'] = 'gpt-4o-mini'
dc['synthetic']['n_pairs_per_topic'] = 50
dc['preprocessing']['tanglish_ratio_min'] = 0.05
dc['preprocessing']['tanglish_ratio_max'] = 0.95
dc['preprocessing']['min_words'] = 10
yaml.safe_dump(dc, open('config/data_config.yaml', 'w'), sort_keys=False)
print(f"Data config: {len(dc['youtube']['target_channels'])} channels, loose filters")

# Model config: bf16 training + memory-reduced batch + 1 epoch
mc = yaml.safe_load(open('config/model_config.yaml'))
mc['training']['num_train_epochs'] = 1
mc['training']['per_device_train_batch_size'] = 2
mc['training']['per_device_eval_batch_size'] = 2
mc['training']['gradient_accumulation_steps'] = 8
mc['training']['fp16'] = False
mc['training']['bf16'] = True
mc['training']['eval_steps'] = 30
mc['training']['save_steps'] = 60
mc['training']['logging_steps'] = 5
mc['training']['save_total_limit'] = 2
mc['callbacks']['memory_log_every_steps'] = 30
mc['callbacks']['sample_generation_every_steps'] = 60
mc['callbacks']['early_stopping_patience'] = 3
mc['tokenizer']['max_seq_length'] = 512
yaml.safe_dump(mc, open('config/model_config.yaml', 'w'), sort_keys=False)
print('Model config: 1 epoch, batch=2, bf16, max_seq=512')

# Patch synthetic generator (JSON mode)
from src.data_collection.synthetic_generator import SyntheticGenerator
def _patched_call(self, prompt):
    obj_prompt = prompt + (
        '\n\nIMPORTANT FORMAT: Return JSON object {"pairs": [{"question":..., "answer":..., "topic":..., "difficulty":...}]}. '
        'Each item must be a JSON OBJECT, NOT a string.'
    )
    resp = self.client.chat.completions.create(
        model=self.model,
        messages=[
            {'role': 'system', 'content': 'You output ONLY valid JSON with structured objects.'},
            {'role': 'user', 'content': obj_prompt},
        ],
        temperature=self.temperature, max_tokens=3000,
        response_format={'type': 'json_object'},
    )
    raw = (resp.choices[0].message.content or '').strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            for v in obj.values():
                if isinstance(v, list):
                    return json.dumps([x for x in v if isinstance(x, dict)], ensure_ascii=False)
    except: pass
    return raw
SyntheticGenerator._call_model = _patched_call

# Patch text cleaner (fast exact dedup)
import src.preprocessing.text_cleaner as tc
def fast_dedup(self, records):
    seen, kept = set(), []
    for rec in records:
        key = ((rec.get('question') or '').strip().lower(),
               (rec.get('answer') or '').strip().lower())
        if key in seen: continue
        seen.add(key); kept.append(rec)
    return kept
tc.TextCleaner._dedup_minhash = fast_dedup

print('All patches applied: synthetic JSON mode + fast dedup')

## 4 — Phase 1A: YouTube scrape (12 channels, ~30-40 min)

In [ ]:
from src.data_collection.youtube_scraper import run as run_scrape
run_scrape('config/data_config.yaml')   # all 12 channels, no limit
!ls -lh data/raw/

## 5 — Phase 1B: Synthetic QA gen (gpt-4o-mini, ~12 min, ~$0.10)

In [ ]:
from src.data_collection.synthetic_generator import run as run_synth
run_synth('config/data_config.yaml', n_per_topic=50)   # 500 pairs

## 6 — Phase 1C: Merge all sources

In [ ]:
from src.data_collection.dataset_merger import merge_all
merged_path, counts = merge_all('config/data_config.yaml')
print('Counts:', counts)
!wc -l data/raw/merged_corpus.jsonl

## 7 — Phase 2: Preprocessing (fast dedup, ~5 min)

In [ ]:
from src.preprocessing.text_cleaner import run as run_clean
from src.preprocessing.tanglish_detector import run as run_tanglish
from src.preprocessing.qa_formatter import format_and_split

Path('data/processed').mkdir(exist_ok=True)
Path('data/final').mkdir(exist_ok=True)

run_clean('config/data_config.yaml',
          input_path='data/raw/merged_corpus.jsonl',
          output_path='data/processed/cleaned.jsonl')

run_tanglish('config/data_config.yaml',
             input_dir='data/processed/cleaned.jsonl',
             output_dir='data/processed/tanglish.jsonl')

stats = format_and_split(
    input_path='data/processed/tanglish.jsonl',
    output_dir='data/final',
    config_path='config/data_config.yaml',
)
print(json.dumps(stats.get('overall', {}), indent=2))

## 8 — 💾 SAVE DATA BACKUP (prevents loss on timeout!)

After this cell runs, **click 'Save Version' → 'Quick Save'** to persist the backup.

In [ ]:
n_train = sum(1 for _ in open('data/final/train.jsonl'))
n_val = sum(1 for _ in open('data/final/val.jsonl'))
n_test = sum(1 for _ in open('data/final/test.jsonl'))
print(f'Dataset: train={n_train}, val={n_val}, test={n_test}, total={n_train+n_val+n_test}')

shutil.make_archive('/kaggle/working/tamiltech-qa-data', 'zip', 'data/final')
sz = Path('/kaggle/working/tamiltech-qa-data.zip').stat().st_size / 1024**2
print(f'Backup zip: /kaggle/working/tamiltech-qa-data.zip ({sz:.2f} MB)')
print()
print('⚠️ MANUAL STEP NOW:')
print('  1. Top-right: Save Version → Quick Save')
print('  2. Wait ~30 sec for save to complete')
print('  3. Continue to next cell only after save is done')

## 9 — Training patches (train_one with all fixes)

In [ ]:
import src.training.trainer as trainer_mod

def patched_train_one(config_name, dataset_dir, model_config_path,
                     base_model_override=None, timestamp=None,
                     sample_prompts=None, merge_after=False):
    from dataclasses import asdict
    import time, torch
    from src.utils import ensure_dir
    from src.utils.seed import seed_everything
    from src.training.model_loader import load_quantized_model
    from src.training.lora_config import get_lora_config, count_trainable_parameters
    from src.training.trainer import load_splits, _formatting_func
    from src.training.callbacks import (LossHistoryCallback, MemoryCallback,
                                         SampleGenerationCallback, make_early_stopping_callback)
    from peft import get_peft_model, prepare_model_for_kbit_training
    from trl import SFTConfig, SFTTrainer

    cfg = load_config(model_config_path)
    seed_everything(cfg.get('seed', 42))
    ts = timestamp or time.strftime('%Y%m%d_%H%M%S')
    run_name = f'{config_name}_{ts}'
    output_root = project_root() / cfg['training'].get('output_root', 'outputs')
    output_dir = ensure_dir(output_root / run_name)

    bundle = load_quantized_model(model_name=base_model_override,
                                   model_config_path=model_config_path, use_4bit=True)
    model, tokenizer = bundle.model, bundle.tokenizer
    max_seq = cfg['tokenizer'].get('max_seq_length', 512)
    tokenizer.model_max_length = max_seq

    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=cfg['training'].get('gradient_checkpointing', True))
    spec = get_lora_config(config_name, model_config_path)
    peft_model = get_peft_model(model, spec.to_peft())
    count_trainable_parameters(peft_model)

    # Cast bf16 trainable params (safety net even though we use bf16 training)
    cast = 0
    for n, p in peft_model.named_parameters():
        if p.requires_grad and p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32); cast += 1
    print(f'Cast {cast} bf16 trainable params to fp32')

    train_ds, eval_ds = load_splits(dataset_dir)
    print(f'Train rows={len(train_ds)} Val rows={len(eval_ds)}')

    tr = cfg['training']
    base = dict(
        output_dir=str(output_dir),
        num_train_epochs=float(tr.get('num_train_epochs', 1)),
        per_device_train_batch_size=int(tr.get('per_device_train_batch_size', 2)),
        per_device_eval_batch_size=int(tr.get('per_device_eval_batch_size', 2)),
        gradient_accumulation_steps=int(tr.get('gradient_accumulation_steps', 8)),
        warmup_ratio=float(tr.get('warmup_ratio', 0.03)),
        learning_rate=float(tr.get('learning_rate', 2e-4)),
        weight_decay=float(tr.get('weight_decay', 0.0)),
        fp16=bool(tr.get('fp16', False)),
        bf16=bool(tr.get('bf16', True)),
        logging_steps=int(tr.get('logging_steps', 5)),
        eval_steps=int(tr.get('eval_steps', 30)),
        save_steps=int(tr.get('save_steps', 60)),
        save_total_limit=int(tr.get('save_total_limit', 2)),
        load_best_model_at_end=bool(tr.get('load_best_model_at_end', True)),
        metric_for_best_model=tr.get('metric_for_best_model', 'eval_loss'),
        greater_is_better=bool(tr.get('greater_is_better', False)),
        lr_scheduler_type=tr.get('lr_scheduler_type', 'cosine'),
        optim=tr.get('optim', 'paged_adamw_8bit'),
        report_to=tr.get('report_to', 'none'),
        run_name=run_name,
        gradient_checkpointing=bool(tr.get('gradient_checkpointing', True)),
    )
    eval_strategy = tr.get('evaluation_strategy', 'steps')
    sft_config = None
    for sk in [{'eval_strategy': eval_strategy}, {'evaluation_strategy': eval_strategy}, {}]:
        try:
            sft_config = SFTConfig(**base, **sk); break
        except TypeError: continue
    if sft_config is None: raise RuntimeError('Could not build SFTConfig')
    for attr, val in [('max_seq_length', max_seq), ('max_length', max_seq), ('packing', False)]:
        if hasattr(sft_config, attr):
            try: setattr(sft_config, attr, val)
            except: pass

    cb_cfg = cfg.get('callbacks', {})
    callbacks = [
        MemoryCallback(every_steps=int(cb_cfg.get('memory_log_every_steps', 30))),
        LossHistoryCallback(),
        make_early_stopping_callback(
            patience=int(cb_cfg.get('early_stopping_patience', 3)),
            threshold=float(cb_cfg.get('early_stopping_threshold', 0.0))),
    ]
    if sample_prompts:
        callbacks.append(SampleGenerationCallback(
            prompts=sample_prompts, tokenizer=tokenizer,
            every_steps=int(cb_cfg.get('sample_generation_every_steps', 60)),
            num_prompts=int(cb_cfg.get('num_sample_prompts', 3))))

    common = dict(model=peft_model, args=sft_config, train_dataset=train_ds,
                  eval_dataset=eval_ds, formatting_func=_formatting_func, callbacks=callbacks)
    try: trainer = SFTTrainer(**common, processing_class=tokenizer)
    except TypeError:
        try: trainer = SFTTrainer(**common, tokenizer=tokenizer)
        except TypeError: trainer = SFTTrainer(**common)

    print(f'Starting training: {run_name}')
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    meta = {'config_name': config_name, 'lora_spec': asdict(spec),
            'base_model': bundle.model_name, 'run_name': run_name, 'output_dir': str(output_dir)}
    (output_dir / 'run_metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print(f'Run complete: {output_dir}')
    return output_dir

trainer_mod.train_one = patched_train_one
print('Patched train_one with: bf16 training + cast fix + modern SFTConfig/SFTTrainer API')

## 10 — Phase 3 Step A: Resolve base model path

In [ ]:
import torch
from pathlib import Path as _Path

USE_HF_LLAMA = True
KAGGLE_LLAMA = _Path('/kaggle/input/llama-3.1/transformers/8b-instruct/1')
if KAGGLE_LLAMA.exists():
    BASE_MODEL = str(KAGGLE_LLAMA)
    print('Using Kaggle-attached Llama-3.1:', BASE_MODEL)
elif USE_HF_LLAMA and os.environ.get('HF_TOKEN'):
    BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
    print('Using HF Llama-3.1-8B-Instruct (will download ~16GB on first use)')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
else:
    BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
    print('Falling back to Mistral:', BASE_MODEL)

# Patch model config with chosen base + bf16 settings
mc = yaml.safe_load(open('config/model_config.yaml'))
mc['base_model']['default'] = BASE_MODEL
mc['training']['fp16'] = False
mc['training']['bf16'] = True
mc['training']['report_to'] = 'none'
yaml.safe_dump(mc, open('config/model_config.yaml', 'w'), sort_keys=False)
print('Model config patched.')

## 11 — Phase 3 Step B: TRAIN!

**⏱️ ~5-6 hours on T4 with bf16. Save Version every 30 min.**

Using `config_name='lora_r8'` for faster training (smaller adapter, less memory).

In [ ]:
from src.training.trainer import train_one, _read_sample_prompts

ds_dir = Path('data/final')
sample_prompts = _read_sample_prompts(ds_dir, k=3)

run_dir = train_one(
    config_name='lora_r8',   # smaller + faster than r16
    dataset_dir=ds_dir,
    model_config_path='config/model_config.yaml',
    sample_prompts=sample_prompts,
    merge_after=False,
)
print('Training output:', run_dir)
!ls -lh $run_dir

## 12 — Save trained model backup (after training)

In [ ]:
# Backup trained adapter
shutil.make_archive('/kaggle/working/tamiltech-qa-model', 'zip', str(run_dir))
sz = Path('/kaggle/working/tamiltech-qa-model.zip').stat().st_size / 1024**2
print(f'Model backup zip: /kaggle/working/tamiltech-qa-model.zip ({sz:.2f} MB)')
print('⚠️ NOW: Save Version (Quick Save) to preserve the trained model')

## 13 — Phase 4: Evaluation

In [ ]:
# Patch eval config to point at our trained adapter
ev = yaml.safe_load(open('config/eval_config.yaml'))
ev['paths']['test_set'] = 'data/final/test.jsonl'
new_models = []
for m in ev['models_to_eval']:
    if m['name'] == 'base_zero_shot':
        m['base_model'] = BASE_MODEL; new_models.append(m)
    elif m['name'] == 'lora_r8':
        m['base_model'] = BASE_MODEL
        m['adapter_path'] = str(run_dir)
        new_models.append(m)
    elif m['name'] == 'lora_r16' and not any(x['name'] == 'lora_r8' for x in ev['models_to_eval']):
        # if lora_r8 not in template, point lora_r16 at our adapter
        m['base_model'] = BASE_MODEL
        m['adapter_path'] = str(run_dir)
        new_models.append(m)
if not any(m['name'] == 'lora_r8' for m in new_models):
    new_models.append({'name': 'lora_r8', 'type': 'lora',
                       'base_model': BASE_MODEL, 'adapter_path': str(run_dir)})
ev['models_to_eval'] = new_models
yaml.safe_dump(ev, open('config/eval_config.yaml', 'w'), sort_keys=False)
print('Eval config patched. Models:', [m['name'] for m in new_models])

from src.evaluation.metrics import evaluate_all_models
metrics = evaluate_all_models(
    eval_config_path='config/eval_config.yaml',
    model_config_path='config/model_config.yaml',
    data_config_path='config/data_config.yaml',
)

import pandas as pd
pd.DataFrame(metrics).T

In [ ]:
# Generate plots
from src.evaluation.visualizer import generate_all
generate_all(eval_cfg_path='config/eval_config.yaml')
from IPython.display import Image, display
for fig in Path('outputs/figures').glob('*.png'):
    display(Image(filename=str(fig)))

## 14 — Phase 5: Push to HuggingFace Hub (optional)

Replace `your-username` with your HF handle.

In [ ]:
HF_USER = 'your-username'   # ← EDIT THIS
DATASET_REPO = f'{HF_USER}/TamilTech-QA'
MODEL_REPO = f'{HF_USER}/TamilTech-QA-Llama3.1-8B-QLoRA'

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

from src.utils.hf_uploader import push_dataset, push_model
push_dataset(Path('data/final'), repo=DATASET_REPO, private=False)
push_model(Path(run_dir), repo=MODEL_REPO, private=False,
           base_model=BASE_MODEL, lora_name='lora_r8',
           dataset_repo=DATASET_REPO,
           metrics_json=Path('outputs/evaluation/metrics_comparison.json'))
print('Done!')
print(f'  Dataset: https://huggingface.co/datasets/{DATASET_REPO}')
print(f'  Model:   https://huggingface.co/{MODEL_REPO}')

## Done!

If something fails mid-way:
1. **Click Save Version** to preserve outputs
2. Restart kernel and re-run cells from the failure point
3. Data + model backups in `/kaggle/working/*.zip` survive Save Version saves